# Проект: Построение витрины данных для Яндекс Такси

**Автор:** Васильчева Светлана

**Дата:** 20.05.2026

**Цель:** Создать автоматизированный ETL-пайплайн для агрегации данных о поездках такси

---

## 📋 Описание проекта

### Бизнес-задача
Финансовая и продуктовая команды Яндекс Такси нуждаются в ежедневных данных о выручке и поведении пассажиров.

### Решение
Построить витрину данных, которая агрегирует информацию по способам оплаты.

### Ключевые метрики
- Количество поездок
- Средняя стоимость поездки
- Средние чаевые
- Суммарная выручка

---

## 🛠 Технологии

| Компонент | Технология |
|-----------|------------|
| Оркестрация | Apache Airflow |
| Обработка | PySpark |
| Хранилище | S3 (Yandex Cloud) |
| База данных | ClickHouse |

---

## 📊 Данные

**Исходная таблица:** `taxi_data.parquet`

| Поле | Описание |
|------|----------|
| `driver_id` | Идентификатор водителя |
| `start_time` | Время начала поездки |
| `end_time` | Время окончания поездки |
| `duration_sec` | Длительность в секундах |
| `distance` | Дистанция поездки |
| `fare` | Стоимость поездки |
| `tips` | Размер чаевых |
| `trip_total` | Общая стоимость (fare + tips + комиссия) |
| `payment_type` | Способ оплаты |

**Результат:** Агрегированная таблица `taxi_payment_summary`

| Поле | Описание |
|------|----------|
| `payment_type` | Способ оплаты |
| `count_trips` | Количество поездок |
| `avg_fare` | Средняя стоимость |
| `avg_tips` | Средние чаевые |
| `revenue` | Суммарная выручка |

---

## 🔧 Шаг 1: Spark-скрипт для обработки данных

**Файл:** scripts/my_spark_job.py

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Создание Spark-сессии
spark = SparkSession.builder \
    .appName("TaxiAggregation") \
    .config("fs.s3a.endpoint", "storage.yandexcloud.net") \
    .config("spark.executor.instances", "1") \
    .config("spark.executor.memory", "1g") \
    .getOrCreate()

# 2. Параметры подключения к ClickHouse
jdbcHostname = "rc1a-3jouval14nne7aun.mdb.yandexcloud.net"
jdbcPort = 8443
jdbcDatabase = "playground_da_20260330_d3b4475e83"
username = "da_20260330_d3b4475e83"
password = "6673f77471ff4dee9ef1246de32b8b6f"  # ⚠️ В продакшене использовать переменные окружения

jdbcUrl = f"jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true"

# 3. Чтение данных из S3
df = spark.read.parquet("s3a://da-plus-dags/project_04/taxi_data.parquet")

print("✅ Данные загружены")
print(f"Количество записей: {df.count()}")

# 4. Агрегация по способу оплаты
result_df = df.groupBy("payment_type").agg(
    F.count("*").alias("count_trips"),
    F.avg("fare").alias("avg_fare"),
    F.avg("tips").alias("avg_tips"),
    F.sum("trip_total").alias("revenue")
)

print("\n📊 Результат агрегации:")
result_df.show()

# 5. Запись в ClickHouse
result_df.write \
    .format("jdbc") \
    .option("url", jdbcUrl) \
    .option("user", username) \
    .option("password", password) \
    .option("dbtable", "taxi_payment_summary") \
    .option("driver", "ru.yandex.clickhouse.ClickHouseDriver") \
    .mode("append") \
    .save()

print("\n✅ Данные успешно записаны в ClickHouse!")

# 6. Остановка Spark-сессии
spark.stop()

ModuleNotFoundError: No module named 'pyspark'

**Что делает скрипт:**

|Шаг  |Действие |
|-----|---------|
|1| Создает Spark-сессию с настройками|
|2| Задает параметры подключения к ClickHouse|
|3| Читает данные из S3 (Parquet)|
|4| Группирует по payment_type и считает метрики|
|5| Записывает результат в ClickHouse|
|6| Завершает сессию|

## 🔄 Шаг 2: DAG для Airflow

**Файл:** dags/taxi_dag.py

In [ ]:
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator
from datetime import datetime

# 1. Настройки DAG
DAG_ID = "taxi_analys"
user = "da_20260330_d3b4475e83"

# 2. Определение DAG
with DAG(
    DAG_ID,
    schedule_interval="@daily",           # Запуск каждый день
    start_date=datetime(2025, 1, 1),      # Дата начала
    catchup=False                         # Не запускать пропущенные дни
) as dag:

    # 3. Проверка наличия файла в S3
    check_s3_file = S3KeySensor(
        task_id="check_s3_file",
        bucket_name="da-plus-dags",
        bucket_key="project_04/taxi_data.parquet",
        aws_conn_id="s3",
        poke_interval=60 * 5,             # Проверка каждые 5 минут
        timeout=60 * 60,                  # Таймаут 1 час
        mode="poke"
    )

    # 4. Запуск Spark-задачи
    run_spark_job = DataprocCreatePysparkJobOperator(
        task_id="run_spark_job",
        cluster_id="c9q4134h5vi546h1e148",
        main_python_file_uri=f"s3a://da-plus-dags/{user}/jobs/my_spark_job.py"
    )

    # 5. Определение порядка выполнения
    check_s3_file >> run_spark_job

print("✅ DAG загружен в Airflow")

**Что делает DAG:**

|Шаг|Task|Описание|
|---|----|--------|
|1|check_s3_file|Ждет появления файла в S3|
|2|run_spark_job|Запускает Spark-скрипт|
|3|Порядок|Сначала проверка, потом запуск|

## 📊 Пример результата
**После выполнения пайплайна в ClickHouse появляется таблица:**

In [ ]:
SELECT * FROM taxi_payment_summary;

**Результат:**

|payment_type|count_trips|avg_fare|avg_tips|revenue|
|------------|-----------|--------|--------|-------|
|card|1,234,567|450.50|52.30|556,000,000|
|cash|987,654|380.20|15.40|375,000,000|
|google_pay|123,456|520.00|65.20|64,200,000|
|apple_pay|89,123|490.00|58.70|43,600,000|
|other|45,678|350.00|10.00|16,000,000|

Результат приведен для примера вывода данных и не является реальными данными